In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "datasets").exists():
    ROOT = ROOT.parent

RAW = ROOT / "datasets" / "raw"
OUT = ROOT / "datasets" / "processed"
REPORTS = ROOT / "reports"
OUT.mkdir(exist_ok=True)
REPORTS.mkdir(exist_ok=True)

reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
products = pd.read_csv(RAW / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
translation = pd.read_csv(RAW / "product_category_name_translation.csv")

reviews["review_text_clean"] = (
    reviews["review_comment_title"].fillna("").astype(str).str.strip()
    + " "
    + reviews["review_comment_message"].fillna("").astype(str).str.strip()
).str.normalize("NFKC").str.replace(r"\s+", " ", regex=True).str.strip()

review_nlp = reviews.loc[
    reviews["review_text_clean"].ne("")
].drop_duplicates(["review_id", "order_id"]).copy()

assert review_nlp["review_text_clean"].ne("").all()
print("Reviews for NLP:", len(review_nlp))

Reviews for NLP: 42687


In [2]:
from transformers import pipeline
from tqdm.auto import tqdm

checkpoint = OUT / "review_sentiment.parquet"
model_name = "clapAI/roberta-base-multilingual-sentiment"

required = {"review_id", "order_id", "sentiment", "sentiment_confidence"}

if checkpoint.exists():
    cached = pd.read_parquet(checkpoint)
    valid = (
        required.issubset(cached.columns)
        and len(cached) == len(review_nlp)
        and cached[["review_id", "order_id"]].reset_index(drop=True).equals(
            review_nlp[["review_id", "order_id"]].reset_index(drop=True)
        )
        and cached["sentiment"].notna().all()
        and cached["sentiment_confidence"].between(0, 1).all()
    )
else:
    valid = False

if valid:
    review_nlp = cached.copy()
    print("Reused validated sentiment checkpoint.")
else:
    classifier = pipeline(
        "text-classification",
        model=model_name,
        device=-1
    )

    texts = review_nlp["review_text_clean"].tolist()
    predictions = []

    for start in tqdm(range(0, len(texts), 32)):
        batch = texts[start:start + 32]
        result = classifier(
            batch,
            batch_size=32,
            truncation=True,
            max_length=256
        )
        predictions.extend(result)

    assert len(predictions) == len(review_nlp)

    review_nlp["sentiment"] = [
        p["label"].lower() for p in predictions
    ]
    review_nlp["sentiment_confidence"] = [
        p["score"] for p in predictions
    ]

    review_nlp.to_parquet(checkpoint, index=False)
    print("Saved sentiment checkpoint.")

print(review_nlp["sentiment"].value_counts())

/opt/anaconda3/envs/retailai-nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  0%|                                                                                                | 0/1334 [00:00<?, ?it/s]

  0%|                                                                                        | 1/1334 [00:01<34:50,  1.57s/it]

  0%|▏                                                                                       | 2/1334 [00:03<34:12,  1.54s/it]

  0%|▏                                                                                       | 3/1334 [00:04<30:47,  1.39s/it]

  0%|▎                                                                                       | 4/1334 [00:05<30:30,  1.38s/it]

... (1,328 outputs left out)

100%|█████████████████████████████████████████████████████████████████████████████████▉| 1333/1334 [14:53:14<00:00,  1.69it/s]

100%|██████████████████████████████████████████████████████████████████████████████████| 1334/1334 [14:53:15<00:00,  1.75it/s]

100%|██████████████████████████████████████████████████████████████████████████████████| 1334/1334 [14:53:15<00:00, 40.18s/it]

Saved sentiment checkpoint.
sentiment
positive    22371
negative    11772
neutral      8544
Name: count, dtype: int64


In [3]:
import re

# Weak-label taxonomy: interpretable, not a validated classifier.
complaint_patterns = {
    "delivery": r"atras|demor|entreg|prazo|chegou|receb|transport",
    "packaging": r"embalag|pacote|caixa|embrulh|proteção",
    "product_quality": r"quebrad|defeit|danific|estragad|qualidade|funciona|travando",
    "wrong_or_missing_item": r"errad|faltando|faltou|diferente|não veio|nao veio",
    "refund_return": r"reembols|devolu|estorn|troca|dinheiro de volta",
    "service": r"atendimento|suporte|vendedor|resposta|contato"
}

# Complaint detection is restricted to low-rated reviews to
# reduce false complaints from positive mentions.
low_rating = review_nlp["review_score"].le(2)
negative_text = review_nlp["sentiment"].eq("negative")
complaint_candidate = low_rating | negative_text

for topic, pattern in complaint_patterns.items():
    review_nlp["complaint_" + topic] = (
        complaint_candidate
        & review_nlp["review_text_clean"].str.contains(
            pattern, case=False, regex=True, na=False
        )
    )

topic_cols = ["complaint_" + t for t in complaint_patterns]
review_nlp["has_detected_complaint"] = review_nlp[topic_cols].any(axis=1)

review_nlp["low_rating"] = low_rating

print("Sentiment by rating:")
print(pd.crosstab(
    review_nlp["review_score"],
    review_nlp["sentiment"],
    normalize="index"
).round(3))

print("\nComplaint topic counts:")
print(review_nlp[topic_cols].sum().sort_values(ascending=False))

Sentiment by rating:


sentiment     negative  neutral  positive
review_score                             
1                0.905    0.078     0.016
2                0.712    0.252     0.036
3                0.390    0.438     0.172
4                0.073    0.407     0.520
5                0.016    0.145     0.838

Complaint topic counts:
complaint_delivery                 8364
complaint_wrong_or_missing_item    1566
complaint_product_quality          1368
complaint_refund_return            1164
complaint_service                  1114
complaint_packaging                 548
dtype: int64


In [4]:
# Keep review-level data at its original grain.
review_business = review_nlp.merge(
    orders[["order_id", "customer_id", "order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one"
)

review_business = review_business.merge(
    customers[
        ["customer_id", "customer_unique_id",
         "customer_city", "customer_state"]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# One order may contain several products and sellers.
# Build order-level product context without multiplying reviews.
item_context = items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left",
    validate="many_to_one"
)

item_context = item_context.merge(
    translation,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

order_context = item_context.groupby("order_id").agg(
    item_count=("order_item_id", "count"),
    unique_products=("product_id", "nunique"),
    unique_sellers=("seller_id", "nunique"),
    unique_categories=("product_category_name_english", "nunique")
).reset_index()

review_business = review_business.merge(
    order_context,
    on="order_id",
    how="left",
    validate="many_to_one"
)

assert len(review_business) == len(review_nlp)

# Attribute a review to a product/seller only when the
# order has exactly one unique product and one unique seller.
single_context = item_context.groupby("order_id").agg(
    product_id=("product_id", "first"),
    seller_id=("seller_id", "first"),
    category=("product_category_name_english", "first")
).reset_index()

review_business = review_business.merge(
    single_context,
    on="order_id",
    how="left",
    validate="many_to_one"
)

ambiguous = (
    review_business["unique_products"].ne(1)
    | review_business["unique_sellers"].ne(1)
)

review_business.loc[
    ambiguous, ["product_id", "seller_id", "category"]
] = None

print("Business dataset:", review_business.shape)
print("Reviews with unambiguous product/seller attribution:",
      review_business["product_id"].notna().sum())

Business dataset: (42687, 30)
Reviews with unambiguous product/seller attribution: 40317


In [5]:
# Minimum sample size avoids ranking tiny groups.
MIN_REVIEWS = 30

category_kpis = (
    review_business.dropna(subset=["category"])
    .groupby("category")
    .agg(
        review_count=("review_id", "size"),
        negative_rate=("low_rating", "mean"),
        average_rating=("review_score", "mean"),
        detected_complaint_rate=("has_detected_complaint", "mean")
    )
    .reset_index()
)

category_kpis = category_kpis.loc[
    category_kpis["review_count"] >= MIN_REVIEWS
].sort_values(
    ["negative_rate", "review_count"],
    ascending=[False, False]
)

seller_kpis = (
    review_business.dropna(subset=["seller_id"])
    .groupby("seller_id")
    .agg(
        review_count=("review_id", "size"),
        negative_rate=("low_rating", "mean"),
        average_rating=("review_score", "mean")
    )
    .reset_index()
)

seller_kpis = seller_kpis.loc[
    seller_kpis["review_count"] >= MIN_REVIEWS
].sort_values(
    ["negative_rate", "review_count"],
    ascending=[False, False]
)

print("Top categories by low-rating rate:")
display(category_kpis.head(10))

print("Top sellers by low-rating rate:")
display(seller_kpis.head(10))

Top categories by low-rating rate:


,category,review_count,negative_rate,average_rating,detected_complaint_rate
30,fashion_male_clothing,58,0.379310,3.293103,0.293103
19,construction_tools_safety,70,0.357143,3.357143,0.314286
57,office_furniture,594,0.348485,3.168350,0.355219
4,audio,160,0.325000,3.487500,0.312500
1,air_conditioning,105,0.314286,3.476190,0.342857
0,agro_industry_and_commerce,64,0.312500,3.421875,0.296875
34,fixed_telephony,101,0.297030,3.485149,0.247525
33,fashion_underwear_beach,51,0.294118,3.666667,0.254902
15,computers_accessories,2698,0.281319,3.601557,0.265011
6,baby,1031,0.277401,3.640155,0.267701


Top sellers by low-rating rate:


,seller_id,review_count,negative_rate,average_rating
280,1ca7077d890b907f89be8c954a02686a,69,0.840580,1.521739
461,2eb70248d66e0e3ef83659f71b244378,129,0.697674,1.976744
1337,8444e55c1f13cd5c179851e5ca5ebd00,46,0.586957,2.413043
1648,a49928bcdf77c55c6d6e05e09a9b4ca5,50,0.580000,2.520000
842,54965bbe3e4f07ae045b90b0b8541f52,40,0.575000,2.550000
1133,710e3548e02bc1d2831dfc4f1b5b14d4,67,0.552239,2.522388
1859,bbad7e518d7af88a0897397ffdca1979,32,0.531250,2.656250
1522,972d0f9cf61b499a4812cf0bfa3ad3c4,38,0.526316,2.710526
595,3c7c4a49ec3c6550809089c6a2ca9370,54,0.518519,2.907407
242,18a349e75d307f4b4cc646a691ed4216,66,0.454545,2.954545


In [6]:
final_path = OUT / "review_product_intelligence.parquet"
review_business.to_parquet(final_path, index=False)

category_path = REPORTS / "review_category_kpis.csv"
seller_path = REPORTS / "review_seller_kpis.csv"
topic_path = REPORTS / "review_complaint_topics.csv"

category_kpis.to_csv(category_path, index=False)
seller_kpis.to_csv(seller_path, index=False)

topic_summary = pd.DataFrame({
    "topic": list(complaint_patterns),
    "matched_reviews": [
        int(review_business["complaint_" + topic].sum())
        for topic in complaint_patterns
    ]
})
topic_summary.to_csv(topic_path, index=False)

assert len(pd.read_parquet(final_path)) == len(review_nlp)

print("PIPELINE COMPLETE")
print("Reviews:", len(review_business))
print("Saved:", final_path)
print("Saved:", category_path)
print("Saved:", seller_path)
print("Saved:", topic_path)
print("Note: Complaint topics are heuristic signals, not verified ground truth.")

PIPELINE COMPLETE
Reviews: 42687
Saved: /Users/adityachaubey/Desktop/RetailAI/datasets/processed/review_product_intelligence.parquet
Saved: /Users/adityachaubey/Desktop/RetailAI/reports/review_category_kpis.csv
Saved: /Users/adityachaubey/Desktop/RetailAI/reports/review_seller_kpis.csv
Saved: /Users/adityachaubey/Desktop/RetailAI/reports/review_complaint_topics.csv
Note: Complaint topics are heuristic signals, not verified ground truth.
